# EDA final do Capítulo 3

Este notebook reproduz, de forma enxuta e independente do EDA original, as evidências exploratórias que permanecem no Capítulo 3 da monografia. Ele parte da base analítica final `data/processed/eda_base.parquet`, valida o recorte utilizado no estudo e gera apenas as três figuras finais do capítulo.

O notebook antigo `notebooks/01_eda_ptbr.ipynb` permanece como referência histórica do projeto, mas não é necessário para executar esta reprodução.


## 1. Configuração

Imports, caminhos relativos ao repositório, parâmetros gráficos e funções auxiliares pequenas usadas pelas três figuras.


In [1]:
from pathlib import Path
import shutil
import sys
import warnings

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "eda_base.parquet"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "capitulo3"
MONOGRAFIA_IMG_DIR = PROJECT_ROOT.parent / "mba-tcc-monografia" / "USPSC-img"

TARGET_H = "t_total_port_stay_h"
DATE_COL = "arrival_port_ts"
PORT_LABEL_COL = "port_display_ref"
EXPECTED_N = 129_625
EXPECTED_START = pd.Timestamp("2023-01-01")
EXPECTED_END = pd.Timestamp("2025-12-31")
REFERENCE_STATS_DAYS = {
    "media_d": 2.96,
    "mediana_d": 1.60,
    "p90_d": 6.80,
    "p95_d": 10.57,
}

COMPONENT_COLUMNS = {
    "t_wait_for_berthing_h": "Espera para atracação",
    "t_operation_h": "Operação atracada",
    "t_post_operation_h": "Pós-operação",
    "t_total_port_stay_h": "Permanência total",
}

OPERATION_LABELS = {
    "op_abastecimento_bunker": "Abastecimento (bunker)",
    "op_arribada": "Arribada",
    "op_carga": "Carga",
    "op_descarga": "Descarga",
    "op_desembarque_passageiros": "Desembarque de passageiros",
    "op_embarque_passageiros": "Embarque de passageiros",
    "op_fundeio": "Fundeio",
    "op_fundeio_ship_to_ship": "Fundeio ship-to-ship",
    "op_navio_estado_com_operacao_comercial": "Navio de Estado com operação comercial",
    "op_navio_estado_marinha_brasil": "Navio de Estado - Marinha do Brasil",
    "op_navio_estado_sem_operacao_comercial": "Navio de Estado sem operação comercial",
    "op_offshore": "Offshore",
    "op_reparo_manutencao": "Reparo/manutenção",
    "op_retirada_residuos_com_operacao_comercial": "Retirada de resíduos com operação comercial",
    "op_retirada_residuos_sem_operacao_comercial": "Retirada de resíduos sem operação comercial",
    "op_solicitacao_certificado": "Solicitação de certificado",
    "op_tipo_operacao_nao_mapeado": "Tipo de operação não mapeado",
}

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLOR_TOTAL = "#2f5d7c"
COLOR_MEDIAN = "#3b6f58"
COLOR_P90 = "#b06f2a"
COLOR_GRID = "#d9dde2"
COLOR_BOX = "#8fb3c8"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def fmt_decimal_pt(value, decimals=2):
    """Formata números com vírgula decimal para leitura das tabelas do capítulo."""
    if pd.isna(value):
        return ""
    return f"{value:,.{decimals}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_axis_days(value, _position=None):
    """Formata eixos em dias sem expor nomes técnicos de colunas."""
    return fmt_decimal_pt(value, 0)


def relpath(path):
    """Mostra caminhos relativos ao repositório quando possível."""
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def save_figure(fig, stem):
    """Salva uma figura em PDF vetorial e PNG 300 dpi no diretório do Capítulo 3."""
    pdf_path = OUTPUT_DIR / f"{stem}.pdf"
    png_path = OUTPUT_DIR / f"{stem}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    return pdf_path, png_path


def stats_for_series(series_h):
    """Calcula estatísticas resumidas em horas e dias a partir de uma série em horas."""
    clean_h = series_h.dropna()
    clean_d = clean_h / 24
    return {
        "n": int(clean_h.shape[0]),
        "media_h": clean_h.mean(),
        "mediana_h": clean_h.median(),
        "p90_h": clean_h.quantile(0.90),
        "p95_h": clean_h.quantile(0.95),
        "p99_h": clean_h.quantile(0.99),
        "media_d": clean_d.mean(),
        "mediana_d": clean_d.median(),
        "p90_d": clean_d.quantile(0.90),
        "p95_d": clean_d.quantile(0.95),
        "p99_d": clean_d.quantile(0.99),
    }


## 2. Carregamento e validação da base

A fonte analítica final é `data/processed/eda_base.parquet`, construída pelo pipeline oficial do projeto. Cada linha representa uma escala portuária elegível.


In [2]:
df = pd.read_parquet(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

missing_required = [
    col for col in [DATE_COL, TARGET_H, PORT_LABEL_COL, *COMPONENT_COLUMNS.keys()]
    if col not in df.columns
]
assert not missing_required, f"Colunas obrigatórias ausentes: {missing_required}"

validation = pd.DataFrame([
    {"item": "Fonte analítica", "valor": relpath(DATA_PATH)},
    {"item": "Escalas portuárias", "valor": f"{len(df):,}".replace(",", ".")},
    {"item": "Menor data de chegada", "valor": df[DATE_COL].min().strftime("%d/%m/%Y %H:%M")},
    {"item": "Maior data de chegada", "valor": df[DATE_COL].max().strftime("%d/%m/%Y %H:%M")},
    {"item": "Número de portos", "valor": f"{df['port'].nunique():,}".replace(",", ".")},
    {"item": "% ausente no target", "valor": fmt_decimal_pt(df[TARGET_H].isna().mean() * 100, 2) + "%"},
    {"item": "Permanências negativas", "valor": int((df[TARGET_H].dropna() < 0).sum())},
])

assert len(df) == EXPECTED_N, f"Total observado ({len(df)}) difere do esperado ({EXPECTED_N})."
assert df[DATE_COL].min().normalize() == EXPECTED_START, "Data inicial diferente do período esperado."
assert df[DATE_COL].max().normalize() == EXPECTED_END, "Data final diferente do período esperado."
assert df[TARGET_H].notna().all(), "Há valores ausentes no target principal."
assert (df[TARGET_H] >= 0).all(), "Há valores negativos no target principal."

validation


,item,valor
0,Fonte analítica,data/processed/eda_base.parquet
1,Escalas portuárias,129.625
2,Menor data de chegada,01/01/2023 00:08
3,Maior data de chegada,31/12/2025 23:00
4,Número de portos,109
5,% ausente no target,"0,00%"
6,Permanências negativas,0


## 3. Estatísticas exploratórias principais

As estatísticas abaixo são recalculadas diretamente da base. Os valores em dias são usados como referência para a redação da monografia e para verificar consistência com a versão final do Capítulo 3.


In [3]:
main_stats = stats_for_series(df[TARGET_H])
reference_check = pd.DataFrame([
    {
        "estatística": "Média",
        "observado_d": main_stats["media_d"],
        "referência_d": REFERENCE_STATS_DAYS["media_d"],
        "diferença_d": main_stats["media_d"] - REFERENCE_STATS_DAYS["media_d"],
    },
    {
        "estatística": "Mediana",
        "observado_d": main_stats["mediana_d"],
        "referência_d": REFERENCE_STATS_DAYS["mediana_d"],
        "diferença_d": main_stats["mediana_d"] - REFERENCE_STATS_DAYS["mediana_d"],
    },
    {
        "estatística": "P90",
        "observado_d": main_stats["p90_d"],
        "referência_d": REFERENCE_STATS_DAYS["p90_d"],
        "diferença_d": main_stats["p90_d"] - REFERENCE_STATS_DAYS["p90_d"],
    },
    {
        "estatística": "P95",
        "observado_d": main_stats["p95_d"],
        "referência_d": REFERENCE_STATS_DAYS["p95_d"],
        "diferença_d": main_stats["p95_d"] - REFERENCE_STATS_DAYS["p95_d"],
    },
])

reference_check_display = reference_check.copy()
for col in ["observado_d", "referência_d", "diferença_d"]:
    reference_check_display[col] = reference_check_display[col].map(lambda x: fmt_decimal_pt(x, 3))

reference_check_display


,estatística,observado_d,referência_d,diferença_d
0,Média,"2,963","2,960","0,003"
1,Mediana,"1,601","1,600","0,001"
2,P90,"6,795","6,800","-0,005"
3,P95,"10,572","10,570","0,002"


In [4]:
component_stats = pd.DataFrame([
    {"variável": label, **stats_for_series(df[col])}
    for col, label in COMPONENT_COLUMNS.items()
])

component_stats_display = component_stats[[
    "variável", "n", "media_h", "mediana_h", "p90_h", "p95_h", "p99_h",
    "media_d", "mediana_d", "p90_d", "p95_d", "p99_d",
]].copy()

for col in [c for c in component_stats_display.columns if c.endswith("_h")]:
    component_stats_display[col] = component_stats_display[col].map(lambda x: fmt_decimal_pt(x, 1))
for col in [c for c in component_stats_display.columns if c.endswith("_d")]:
    component_stats_display[col] = component_stats_display[col].map(lambda x: fmt_decimal_pt(x, 2))

component_stats_display


,variável,n,media_h,mediana_h,p90_h,p95_h,p99_h,media_d,mediana_d,p90_d,p95_d,p99_d
0,Espera para atracação,129625,"8,5","0,0","6,7","38,4","223,6","0,35","0,00","0,28","1,60","9,32"
1,Operação atracada,129625,"61,9","34,8","139,3","210,9","453,6","2,58","1,45","5,81","8,79","18,90"
2,Pós-operação,129625,"0,7","0,0","0,0","0,0","12,7","0,03","0,00","0,00","0,00","0,53"
3,Permanência total,129625,"71,1","38,4","163,1","253,7","527,9","2,96","1,60","6,80","10,57","21,99"


## 4. Figura 1 - Distribuição da permanência e de seus componentes

A visualização abaixo é limitada ao P99 apenas para leitura gráfica. Nenhuma observação é removida da base analítica nem dos cálculos das tabelas anteriores.


In [5]:
visual_limit_d = (df[TARGET_H] / 24).quantile(0.99)
bins = np.linspace(0, visual_limit_d, 55)

fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.0), sharex=True)
axes = axes.ravel()

for ax, (col, label) in zip(axes, COMPONENT_COLUMNS.items()):
    values_d = df[col].dropna() / 24
    shown = values_d[values_d <= visual_limit_d]
    ax.hist(shown, bins=bins, color=COLOR_TOTAL, alpha=0.82, edgecolor="white", linewidth=0.35)
    median_d = values_d.median()
    p90_d = values_d.quantile(0.90)
    ax.axvline(median_d, color=COLOR_MEDIAN, linestyle="--", linewidth=1.2, label="Mediana")
    ax.axvline(p90_d, color=COLOR_P90, linestyle=":", linewidth=1.4, label="P90")
    ax.set_title(label)
    ax.set_xlim(0, visual_limit_d)
    ax.grid(axis="y", color=COLOR_GRID, linewidth=0.6, alpha=0.8)
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_axis_days))

axes[0].set_ylabel("Escalas")
axes[2].set_ylabel("Escalas")
axes[2].set_xlabel("Dias")
axes[3].set_xlabel("Dias")
axes[0].legend(frameon=False, loc="upper right")

fig.tight_layout()
fig1_pdf, fig1_png = save_figure(fig, "cap3_eda_distribuicao_permanencia")

pd.DataFrame([{
    "figura": "Distribuição da permanência e componentes",
    "limite_visual_p99_d": fmt_decimal_pt(visual_limit_d, 2),
    "pdf": relpath(fig1_pdf),
    "png": relpath(fig1_png),
}])


,figura,limite_visual_p99_d,pdf,png
0,Distribuição da permanência e componentes,"21,99",outputs/capitulo3/cap3_eda_distribuicao_perman...,outputs/capitulo3/cap3_eda_distribuicao_perman...


## 5. Figura 2 - Heterogeneidade entre portos

Os 15 portos são selecionados exclusivamente por volume de escalas. Após essa seleção, a ordenação da figura usa o P90 para facilitar a leitura da heterogeneidade.


In [6]:
port_summary = (
    df.groupby(PORT_LABEL_COL, dropna=False)[TARGET_H]
    .agg(
        escalas="size",
        mediana_h="median",
        p90_h=lambda s: s.quantile(0.90),
    )
    .reset_index()
)
port_summary["mediana_d"] = port_summary["mediana_h"] / 24
port_summary["p90_d"] = port_summary["p90_h"] / 24

port_top15 = port_summary.nlargest(15, "escalas").copy()
port_top15 = port_top15.sort_values("p90_d", ascending=True).reset_index(drop=True)

port_top15_display = port_top15[[PORT_LABEL_COL, "escalas", "mediana_d", "p90_d"]].copy()
port_top15_display = port_top15_display.rename(columns={PORT_LABEL_COL: "porto"})
for col in ["mediana_d", "p90_d"]:
    port_top15_display[col] = port_top15_display[col].map(lambda x: fmt_decimal_pt(x, 2))
port_top15_display["escalas"] = port_top15_display["escalas"].map(lambda x: f"{x:,}".replace(",", "."))

port_top15_display


,porto,escalas,mediana_d,p90_d
0,BRARE - AREIA BRANCA - RN,3.321,"0,22","0,43"
1,BRIOA001 - ITAPOÁ - SC,1.903,"0,86","1,48"
2,BRRJ022 - SÃO JOÃO DA BARRA - RJ,10.750,"0,97","3,70"
3,BRSSA - SALVADOR - BA,2.258,"0,64","3,92"
4,BRSSO002 - SÃO SEBASTIÃO - SP,2.007,"1,49","4,03"
5,BRMEA001 - MACAÉ - RJ,3.308,"1,20","4,09"
6,BRPNG - PARANAGUÁ - PR,7.495,"1,29","5,62"
7,BRIQI - SÃO LUIZ - MA,2.326,"2,54","6,20"
8,BRVDC - BARCARENA - PA,2.493,"2,08","6,30"
9,BRRIG - SÃO JOSÉ DO NORTE - RS,5.061,"1,87","6,93"


In [7]:
y = np.arange(len(port_top15))

fig, ax = plt.subplots(figsize=(7.2, 5.6))
ax.hlines(y, port_top15["mediana_d"], port_top15["p90_d"], color="#9aa6ad", linewidth=1.4)
ax.scatter(port_top15["mediana_d"], y, color=COLOR_MEDIAN, s=28, label="Mediana", zorder=3)
ax.scatter(port_top15["p90_d"], y, color=COLOR_P90, s=28, label="P90", zorder=3)

ax.set_yticks(y)
ax.set_yticklabels(port_top15[PORT_LABEL_COL])
ax.set_xlabel("Permanência total (dias)")
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: fmt_decimal_pt(value, 1)))
ax.grid(axis="x", color=COLOR_GRID, linewidth=0.7, alpha=0.9)
ax.legend(frameon=False, loc="lower right")

fig.tight_layout()
fig2_pdf, fig2_png = save_figure(fig, "cap3_eda_portos_mediana_p90")

pd.DataFrame([{
    "figura": "Mediana e P90 por porto",
    "portos": len(port_top15),
    "critério_de_seleção": "15 maiores volumes de escalas",
    "pdf": relpath(fig2_pdf),
    "png": relpath(fig2_png),
}])


,figura,portos,critério_de_seleção,pdf,png
0,Mediana e P90 por porto,15,15 maiores volumes de escalas,outputs/capitulo3/cap3_eda_portos_mediana_p90.pdf,outputs/capitulo3/cap3_eda_portos_mediana_p90.png


## 6. Figura 3 - Heterogeneidade por tipo de operação

Os tipos de operação são tratados como flags independentes `op_*`, preservando a natureza multirrótulo da base. Assim, uma mesma escala pode contribuir para mais de uma operação quando sua descrição operacional indicar múltiplos tipos.

Para manter a figura legível, o boxplot exibe apenas operações com pelo menos 1% da base. A tabela de apoio abaixo mantém todas as operações mapeadas.


In [8]:
operation_columns = [col for col in df.columns if col.startswith("op_")]
operation_rows = []

for col in operation_columns:
    mask = df[col].fillna(False).astype(bool)
    values_d = df.loc[mask, TARGET_H] / 24
    operation_rows.append({
        "operação": OPERATION_LABELS.get(col, col.replace("op_", "").replace("_", " ").title()),
        "coluna": col,
        "escalas": int(mask.sum()),
        "percentual_base": mask.mean() * 100,
        "mediana_d": values_d.median(),
        "p90_d": values_d.quantile(0.90),
    })

operation_summary = pd.DataFrame(operation_rows).sort_values("escalas", ascending=False).reset_index(drop=True)
operation_summary_display = operation_summary.copy()
for col in ["percentual_base", "mediana_d", "p90_d"]:
    operation_summary_display[col] = operation_summary_display[col].map(lambda x: fmt_decimal_pt(x, 2))
operation_summary_display["escalas"] = operation_summary_display["escalas"].map(lambda x: f"{x:,}".replace(",", "."))

operation_summary_display[["operação", "escalas", "percentual_base", "mediana_d", "p90_d"]]


,operação,escalas,percentual_base,mediana_d,p90_d
0,Carga,63.445,"48,95","1,62","6,22"
1,Descarga,57.279,"44,19","1,40","6,10"
2,Offshore,33.700,"26,00","1,40","6,20"
3,Abastecimento (bunker),3.780,"2,92","0,97","4,98"
4,Desembarque de passageiros,2.484,"1,92","0,49","4,99"
5,Embarque de passageiros,2.471,"1,91","0,49","5,08"
6,Solicitação de certificado,1.885,"1,45","0,58","8,17"
7,Fundeio,1.305,"1,01","1,75","10,36"
8,Arribada,432,"0,33","2,11","6,65"
9,Reparo/manutenção,317,"0,24","3,82","15,64"


In [9]:
MIN_OPERATION_SHARE = 1.0
operation_plot = operation_summary[operation_summary["percentual_base"] >= MIN_OPERATION_SHARE].copy()
operation_plot = operation_plot.sort_values("p90_d", ascending=True).reset_index(drop=True)
operation_visual_limit_d = (df[TARGET_H] / 24).quantile(0.99)

boxplot_data = []
boxplot_labels = []
for _, row in operation_plot.iterrows():
    mask = df[row["coluna"]].fillna(False).astype(bool)
    values = (df.loc[mask, TARGET_H] / 24).clip(upper=operation_visual_limit_d)
    boxplot_data.append(values.to_numpy())
    boxplot_labels.append(row["operação"])

fig, ax = plt.subplots(figsize=(7.2, 5.4))
box = ax.boxplot(
    boxplot_data,
    vert=False,
    tick_labels=boxplot_labels,
    showfliers=False,
    patch_artist=True,
    widths=0.62,
    medianprops={"color": COLOR_MEDIAN, "linewidth": 1.5},
    boxprops={"facecolor": COLOR_BOX, "edgecolor": "#4f6570", "linewidth": 0.8},
    whiskerprops={"color": "#4f6570", "linewidth": 0.8},
    capprops={"color": "#4f6570", "linewidth": 0.8},
)

ax.set_xlabel("Permanência total (dias)")
ax.set_xlim(0, operation_visual_limit_d)
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: fmt_decimal_pt(value, 1)))
ax.grid(axis="x", color=COLOR_GRID, linewidth=0.7, alpha=0.9)

fig.tight_layout()
fig3_pdf, fig3_png = save_figure(fig, "cap3_eda_operacoes")

pd.DataFrame([{
    "figura": "Distribuição por tipo de operação",
    "operações_na_figura": len(operation_plot),
    "filtro_visual": f"operações com ao menos {fmt_decimal_pt(MIN_OPERATION_SHARE, 1)}% da base",
    "limite_visual_p99_d": fmt_decimal_pt(operation_visual_limit_d, 2),
    "pdf": relpath(fig3_pdf),
    "png": relpath(fig3_png),
}])


,figura,operações_na_figura,filtro_visual,limite_visual_p99_d,pdf,png
0,Distribuição por tipo de operação,8,"operações com ao menos 1,0% da base","21,99",outputs/capitulo3/cap3_eda_operacoes.pdf,outputs/capitulo3/cap3_eda_operacoes.png


## 7. Resumo dos outputs

Os arquivos abaixo são os artefatos finais esperados para inserção no Capítulo 3. Quando o repositório da monografia existir como diretório irmão, os PDFs também serão copiados para `../mba-tcc-monografia/USPSC-img/`.


In [10]:
figure_outputs = pd.DataFrame([
    {
        "figura": "cap3_eda_distribuicao_permanencia",
        "caminho_pdf": relpath(fig1_pdf),
        "caminho_png": relpath(fig1_png),
        "objetivo_metodológico": "Evidenciar assimetria, longa cauda e decomposição da permanência.",
    },
    {
        "figura": "cap3_eda_portos_mediana_p90",
        "caminho_pdf": relpath(fig2_pdf),
        "caminho_png": relpath(fig2_png),
        "objetivo_metodológico": "Evidenciar heterogeneidade entre portos de maior volume.",
    },
    {
        "figura": "cap3_eda_operacoes",
        "caminho_pdf": relpath(fig3_pdf),
        "caminho_png": relpath(fig3_png),
        "objetivo_metodológico": "Evidenciar heterogeneidade por tipos de operação multirrótulo.",
    },
])

copied_to_monografia = []
if MONOGRAFIA_IMG_DIR.exists():
    for pdf_path in [fig1_pdf, fig2_pdf, fig3_pdf]:
        destination = MONOGRAFIA_IMG_DIR / pdf_path.name
        shutil.copy2(pdf_path, destination)
        copied_to_monografia.append(relpath(destination))

if copied_to_monografia:
    print("PDFs copiados para o repositório da monografia:")
    for path in copied_to_monografia:
        print(f"- {path}")
else:
    print("Repositório irmão mba-tcc-monografia/USPSC-img não encontrado; PDFs mantidos apenas em outputs/capitulo3/.")

figure_outputs


Repositório irmão mba-tcc-monografia/USPSC-img não encontrado; PDFs mantidos apenas em outputs/capitulo3/.


,figura,caminho_pdf,caminho_png,objetivo_metodológico
0,cap3_eda_distribuicao_permanencia,outputs/capitulo3/cap3_eda_distribuicao_perman...,outputs/capitulo3/cap3_eda_distribuicao_perman...,"Evidenciar assimetria, longa cauda e decomposi..."
1,cap3_eda_portos_mediana_p90,outputs/capitulo3/cap3_eda_portos_mediana_p90.pdf,outputs/capitulo3/cap3_eda_portos_mediana_p90.png,Evidenciar heterogeneidade entre portos de mai...
2,cap3_eda_operacoes,outputs/capitulo3/cap3_eda_operacoes.pdf,outputs/capitulo3/cap3_eda_operacoes.png,Evidenciar heterogeneidade por tipos de operaç...
